In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path('data') / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()


In [ ]:
df['Churn'].value_counts()

In [ ]:
df['SeniorCitizen'].value_counts()

In [ ]:
df[df['TotalCharges']=='']

In [ ]:
df['TotalCharges']=df['TotalCharges'].replace('','0')

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()

In [ ]:
df.info()

In [ ]:
df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]

In [ ]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
df['TotalCharges'].dtype

In [ ]:
df['TotalCharges'].isnull().sum()

In [ ]:
df['customerID'].duplicated().sum()

In [ ]:
df.info()

In [ ]:
df['Contract'].value_counts()

In [ ]:
df['Churn'].unique()

In [ ]:
df['TotalCharges'].unique()

In [ ]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [ ]:
df['Churn'].value_counts()

In [ ]:
df.select_dtypes(include=['object', 'string']).columns

In [ ]:
df=df.drop('customerID', axis=1)

In [ ]:
for col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    print(col, df[col].unique())

In [ ]:
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
df['Partner'] = df['Partner'].map({'No': 0, 'Yes': 1})
df['Dependents'] = df['Dependents'].map({'No': 0, 'Yes': 1})
df['PhoneService'] = df['PhoneService'].map({'No': 0, 'Yes': 1})
df['PaperlessBilling'] = df['PaperlessBilling'].map({'No': 0, 'Yes': 1})

In [ ]:
df[['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']].head()

In [ ]:
df['gender'].head()

In [ ]:
df['Churn'].head()

In [ ]:
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                   'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train.shape
X_test.shape
y_train.shape
y_test.shape

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf))

In [ ]:
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

top_features = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_features.index, top_features.values)
plt.xlabel('Importance')
plt.title('Top 10 Features Driving Churn Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()